# Laser Marking with Sketches (Database)

This notebook uses the Poetry-managed environment. All dependencies from `pyproject.toml` are available.

A **sketch** places geometries against each other the way a 2D CAD drawing does. Some geometries are explicit shapes, others are computed from constraints on the geometries they reference, and the ones carrying marking parameters are marked by the laser. Sketches replace recipes.

This notebook stores the sketch in the datastore as a **sketch plan**, loads it back, then solves and marks it. To mark a sketch without storing it, see `laser_marking_inline.ipynb`.

In [ ]:
# Verify we're using the correct environment
import sys
print(f"Python: {sys.executable}")

In [ ]:
from femtum_sdk.api.contracts.v1.instruments.health_pb2 import HealthRequest
from femtum_sdk.api import DatastoreApi, InstrumentApi

datastore_host = "localhost:5208"
instrument_host = "localhost:5210"

with DatastoreApi(host=datastore_host) as datastore_api:
    datastore_health_response = datastore_api.health.GetHealth(HealthRequest())
    print("Datastore Health:")
    print(datastore_health_response)

with InstrumentApi(host=instrument_host) as instrument_api:
    instrument_health_response = instrument_api.health.GetHealth(HealthRequest())
    print("Instrument Health:")
    print(instrument_health_response)

## Build a Sketch

The sketch below holds four geometries, all in microns:

- `dot`: an explicit point marked with a single pulse.
- `waveguide`: an explicit line string without marking parameters. It is never marked and only serves as a reference.
- `cut`: a line string computed as the offset of `waveguide`, 10 µm to its right, marked with a pulse train.
- `pad_fill`: an explicit polygon hatched with a cross pattern.

In [ ]:
from femtum_sdk.api.contracts.v1.dto.geometry_pb2 import (
    Coordinate,
    GeometryDistanceUnit,
    LineString,
    Point,
    Polygon,
    Shape,
)
from femtum_sdk.api.contracts.v1.dto.hatching_pb2 import (
    HatchBidirectional,
    HatchContour,
    HatchContourAfterFill,
    HatchCross,
    HatchingParameters,
    HatchOrder,
    HatchPattern,
    HatchShape,
)
from femtum_sdk.api.contracts.v1.dto.marking_parameters_pb2 import (
    LaserMarkingParameters,
    MarkingParameters,
    PulseFrequency,
    PulseFrequencyMaximum,
    PulseLength,
    PulseLengthHalfPeriod,
    PulseParameters,
    SinglePulse,
    TrainPulse,
)
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import (
    AngleMeasure,
    Degrees,
    DistanceMeasure,
    EnergyMeasure,
    Nanojoules,
    Microns,
)
from femtum_sdk.api.contracts.v1.dto.sketch_pb2 import (
    ConstrainedShape,
    OffsetConstraint,
    Sketch,
    SketchConstraint,
    SketchGeometry,
    SketchLineString,
)

microns = GeometryDistanceUnit(distance=Microns)

# At the maximum pulse frequency the laser power is energy x frequency, and it must fall inside the range of the laser energy calibration.
pulse_energy = EnergyMeasure(value=100, unit=Nanojoules)

single_pulse_marking = MarkingParameters(
    laser=LaserMarkingParameters(
        laser_energy=pulse_energy,
        pulse=PulseParameters(single_pulse=SinglePulse()),
    )
)

pulse_train_marking = MarkingParameters(
    laser=LaserMarkingParameters(
        laser_energy=pulse_energy,
        pulse=PulseParameters(
            pulse_train=TrainPulse(
                frequency=PulseFrequency(maximum=PulseFrequencyMaximum()),
                length=PulseLength(half_period=PulseLengthHalfPeriod()),
            )
        ),
    )
)

sketch = Sketch(
    geometries=[
        SketchGeometry(
            id="dot",
            name="Single pulse",
            shape=Shape(point=Point(x=-50, y=0, unit=microns)),
            marking_parameters=single_pulse_marking,
        ),
        SketchGeometry(
            id="waveguide",
            name="Waveguide",
            shape=Shape(
                line_string=LineString(
                    unit=microns,
                    path=[Coordinate(x=0, y=0), Coordinate(x=200, y=0)],
                )
            ),
        ),
        SketchGeometry(
            id="cut",
            name="Cut beside the waveguide",
            constrained=ConstrainedShape(
                line_string=SketchLineString(),
                constraints=[
                    SketchConstraint(
                        id="10um_right_of_waveguide",
                        offset=OffsetConstraint(
                            reference_id="waveguide",
                            distance=DistanceMeasure(value=10, unit=Microns),
                        ),
                    )
                ],
            ),
            marking_parameters=pulse_train_marking,
        ),
        SketchGeometry(
            id="pad_fill",
            name="Pad",
            hatch=HatchShape(
                polygon=Polygon(
                    path=LineString(
                        unit=microns,
                        path=[
                            Coordinate(x=0, y=50),
                            Coordinate(x=100, y=50),
                            Coordinate(x=100, y=100),
                            Coordinate(x=0, y=100),
                            Coordinate(x=0, y=50),
                        ],
                    )
                ),
                parameters=HatchingParameters(
                    pattern=HatchPattern(
                        cross=HatchCross(
                            spacing=DistanceMeasure(value=5, unit=Microns),
                            angle=AngleMeasure(value=45, unit=Degrees),
                            padding=DistanceMeasure(value=2, unit=Microns),
                        )
                    ),
                    order=HatchOrder(bidirectional=HatchBidirectional()),
                    contour=HatchContour(after_fill=HatchContourAfterFill()),
                ),
            ),
            marking_parameters=pulse_train_marking,
        ),
    ]
)

## Store the Sketch as a Sketch Plan

A sketch plan keeps the sketch in the datastore so it can be edited and marked later. The datastore does not solve the sketch, so a sketch the solver would refuse is stored all the same.

In [ ]:
from femtum_sdk.api.contracts.v1.storage.sketch_plan_pb2 import CreateSketchPlanRequest

with DatastoreApi(host=datastore_host) as datastore_api:
    created = datastore_api.sketch_plan.Create(CreateSketchPlanRequest(name="Laser Marking Example", sketch=sketch))

sketch_plan_id = created.sketch_plan.id
print(f"Stored sketch plan {sketch_plan_id}")

## Load the Sketch Plan

`FindById` answers with nothing at all, rather than an error, when the identifier does not match, so check that the plan is present before using it.

In [ ]:
from femtum_sdk.api.contracts.v1.storage.sketch_plan_pb2 import FindSketchPlanByIdRequest

with DatastoreApi(host=datastore_host) as datastore_api:
    found = datastore_api.sketch_plan.FindById(FindSketchPlanByIdRequest(id=sketch_plan_id))

if not found.HasField("sketch_plan"):
    raise LookupError(f"Sketch plan {sketch_plan_id} not found")

stored_sketch = found.sketch_plan.sketch
print(f"{found.sketch_plan.name}: {len(stored_sketch.geometries)} geometries")

## Solve the Stored Sketch

Solving turns the sketch into the marking geometries of an execution plan. It never fires the laser and needs no calibration, so it is safe to call on every change.

Only geometries carrying marking parameters come back, in sketch order: `waveguide` is absent, `cut` is placed 10 µm from it, and the hatch comes back as one geometry per shape of its fill. A sketch the solver refuses is reported as an invalid argument naming the geometries or constraints at fault.

In [ ]:
from femtum_sdk.api.contracts.v1.instruments.sketch_solver_pb2 import SolveSketchRequest, SolveSketchResponse

with InstrumentApi(host=instrument_host) as instrument_api:
    solved: SolveSketchResponse = instrument_api.sketch_solver.Solve(SolveSketchRequest(sketch=stored_sketch))

print(f"{len(solved.marking_geometries)} marking geometries")
for marking_geometry in solved.marking_geometries[:3]:
    print(marking_geometry.shape)

## Rehearse the Plan

`Plan` runs every preparation step of a real run (energy to power conversion, calibrations) and reports what would happen, **without firing the laser**. A rehearsal leaves no log in the datastore.

In [ ]:
from femtum_sdk.api.contracts.v1.dto.execution_plan_pb2 import ExecutionPlan
from femtum_sdk.api.contracts.v1.instruments.laser_execution_pb2 import LaserExecutionFeatures
from femtum_sdk.api.contracts.v1.instruments.plan_execution_pb2 import (
    ExecutionPlanSelector,
    LaserEnergyCalibrationLatest,
    LaserEnergyCalibrationSelector,
    PlanExecutionOptions,
    PlanExecutionRequest,
    PlanExecutionResponse,
)

execution_plan = ExecutionPlan(
    name="Laser Marking Example",
    marking_geometries=solved.marking_geometries,
)
plan_options = PlanExecutionOptions(
    laser_energy_calibration_selector=LaserEnergyCalibrationSelector(latest=LaserEnergyCalibrationLatest()),
    features=LaserExecutionFeatures(take_pictures=True),
)

with InstrumentApi(host=instrument_host) as instrument_api:
    rehearsal: PlanExecutionResponse = instrument_api.plan_execution.Plan(
        PlanExecutionRequest(plan=ExecutionPlanSelector(plan=execution_plan), options=plan_options)
    )

print(rehearsal.log)

## Mark the Plan

> **Warning:** this cell **fires the laser** on the component in the machine. Rehearse the plan first and make sure the machine is ready.

The laser is a safety switch that no other operation flips for you, and a run is refused while it is off, so the cell enables it first.

In [ ]:
from femtum_sdk.api.contracts.v1.instruments.femtosecond_laser_pb2 import EnableLaserRequest, IsEnabledRequest
from femtum_sdk.api.contracts.v1.instruments.plan_execution_pb2 import ExecutePlanRequest, ExecutePlanResponse

with InstrumentApi(host=instrument_host) as instrument_api:
    instrument_api.femtosecond_laser.Enable(EnableLaserRequest(enable=True))
    assert instrument_api.femtosecond_laser.IsEnabled(IsEnabledRequest()).enabled

    execution: ExecutePlanResponse = instrument_api.plan_execution.Execute(
        ExecutePlanRequest(plan=ExecutionPlanSelector(plan=execution_plan), options=plan_options)
    )

print(execution.log)